# Test Fine-tuning Results
Test the fine-tuned LIANet Creoss region performance to the local model perfomance

In [2]:
# import os
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from rasterio.windows import Window
# Add src to path
sys.path.insert(0, '/home/user/src')
import rasterio as rio
from datasets import BuildingCoverageRaster
from models.models_finetune import UNet, MicroUNet
from settings import *



from torchmetrics import MetricCollection

from metrics import multiclass_segmentation_metrics
# Setup device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [3]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

pretrained_model_path = "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01"


def load_model(CKPT_PATH, other_task):
        # # Get config from checkpoint directory
    # finetune_config_path = os.path.join(os.path.dirname(CKPT_PATH), "config.json")
    # if os.path.exists(finetune_config_path):
    #     with open(finetune_config_path, 'r') as f:
    #         config = json.load(f)
    #     model_type = config.get("model_type", "replace_final_block")


    # Create DownstreamModel instance - CR finetuning
    # print(f"Loading DownstreamModel with adaptation strategy '{model_type}'...")
    model_finetune = DownstreamModel(
        model_path=pretrained_model_path,
        checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
        adaption_strategy="replace_final_block",
        num_classes=num_classes[other_task],
        activation="none"
    )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    print("✓ Model loaded successfully")
    return model_finetune

In [4]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import regression_metrics


all_tasks_list = {
    "BFPDensity_joint_T31TFM",
    "BFPDensity_joint_T32ULU",
}

BATCH_SIZE = 16
NUM_WORKERS = 8

results_rows = []


for Target_region in all_tasks_list:
    other_tasks = all_tasks_list - {Target_region}

    val_dataset = BuildingCoverageRaster(
        top_dir=TOP_DIR[Target_region],
        s2_tiles=s2_tiles[Target_region],
        labels=labels[Target_region],
        train_val_key="val",
    )

    dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    for source_region in other_tasks:
        model_dir = (
            f"/home/user/results_local/finetuning_results_BFPDensity/{source_region}/LIANet_lr3e-05_batchsize8_nonburned"
        )

        if not os.path.exists(model_dir):
            print(f"Model directory not found: {model_dir}. Skipping...")
            continue

        for run_name in sorted(os.listdir(model_dir)):
            ckpt_path = os.path.join(model_dir, run_name, "last.pt")

            if not os.path.exists(ckpt_path):
                print(f"Model directory not found: {model_dir}. Skipping...")
                continue

            model = load_model(ckpt_path, source_region)
            model.eval()

            list_of_metrics, _ = regression_metrics()

            metrictracker = MetricCollection(list_of_metrics).to(device)

            with torch.no_grad():
                for batch in tqdm(
                    dataloader,
                    desc=f"{Target_region} | {source_region} | {run_name}",
                    leave=False,
                ):
                    x = batch["x_s2"].to(device)
                    y = batch["y_s2"].to(device)
                    label = batch["label"].to(device)
                    delta_days = batch["delta_days"].to(device)
                    target_tile = Target_region.split("_")[-1]
                    region_idx = 1 if target_tile == "T32ULU" else 2 if target_tile == "T31TFM" else None
                    _, pred = model(
                        delta_days,
                        x,
                        y,
                        torch.tensor([region_idx], device=device),
                    )

                    pred = getattr(pred, "output", pred)

                    if pred.dim() == 3:
                        pred = pred.unsqueeze(1)

                    metrictracker.update(pred.squeeze(), label)

            results = metrictracker.compute()

            row = {
                "target_region": Target_region,
                "source_region": source_region,
                "seed_or_run": run_name,
                "checkpoint_path": ckpt_path,
            }

            for metric_name, metric_value in results.items():
                row[metric_name] = float(metric_value.detach().cpu())

                results_rows.append(row)

            del model
            del metrictracker
            torch.cuda.empty_cache()

    del dataloader
    del val_dataset
    torch.cuda.empty_cache()


df_results = pd.DataFrame(results_rows)

df_results.to_csv(
    "BFPDensity_cr_transfer_results_all_target_regions.csv",
    index=False,
)

df_results

516 val samples loaded from /home/user/data_shared/T32ULU_val_samples_10perc.json
✓ Model loaded successfully


✓ Model loaded successfully


✓ Model loaded successfully


✓ Model loaded successfully


✓ Model loaded successfully


...

✓ Model loaded successfully


✓ Model loaded successfully


✓ Model loaded successfully


✓ Model loaded successfully


,target_region,source_region,seed_or_run,checkpoint_path,mae,mse
0,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-15-30,/home/user/results_local/finetuning_results_BF...,0.127767,0.053299
1,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-15-30,/home/user/results_local/finetuning_results_BF...,0.127767,0.053299
2,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-19-52,/home/user/results_local/finetuning_results_BF...,0.129249,0.053772
3,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-19-52,/home/user/results_local/finetuning_results_BF...,0.129249,0.053772
4,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-24-10,/home/user/results_local/finetuning_results_BF...,0.130519,0.054674
5,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-24-10,/home/user/results_local/finetuning_results_BF...,0.130519,0.054674
6,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-28-26,/home/user/results_local/finetuning_results_BF...,0.131667,0.053438
7,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-28-26,/home/user/results_local/finetuning_results_BF...,0.131667,0.053438
8,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-32-47,/home/user/results_local/finetuning_results_BF...,0.130100,0.053490
9,BFPDensity_joint_T32ULU,BFPDensity_joint_T31TFM,2026-05-11_14-32-47,/home/user/results_local/finetuning_results_BF...,0.130100,0.053490


In [5]:
metadata_cols = [
    "target_region",
    "source_region",
    "seed_or_run",
    "checkpoint_path",
]

metric_cols = [
    col for col in df_results.columns
    if col not in metadata_cols
]

# Average over seeds/runs
df_seed_avg = (
    df_results
    .groupby(["target_region", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# One result per target region
df_target_avg = (
    df_seed_avg
    .groupby(["target_region"], as_index=False)[metric_cols]
    .mean()
)

# Final result over all target regions
df_final_avg = (
    df_target_avg[metric_cols]
    .mean()
    .to_frame()
    .T
)

df_target_avg.to_csv("BFPDensity_transfer_results_avg_per_target_region.csv", index=False)
df_final_avg.to_csv("BFPDensity_transfer_results_final_avg_all_regions.csv", index=False)

df_target_avg, df_final_avg

(             target_region       mae       mse
 0  BFPDensity_joint_T31TFM  0.126847  0.051207
 1  BFPDensity_joint_T32ULU  0.129861  0.053735,
         mae       mse
 0  0.128354  0.052471)

In [6]:
import json
from models.models_finetune import DownstreamModel
from models.LIANet import LIANetLight
import omegaconf, hydra

# pretrained_model_path = "/home/user/results_shared/fourier_learned_4regions/2026-03-18_19-06-01"

pretrained_model_path_list = {
    "T31TFM": "/home/user/results_shared/fourier_learned_T31TFM/2026-04-08_00-13-56",
    "T32ULU": "/home/user/results_shared/fourier_learned_T32ULU/2026-04-04_09-21-54",
}

def load_model(CKPT_PATH, other_task):

    pretrained_model_path = pretrained_model_path_list[other_task.split("_")[-1]]
    model_finetune = DownstreamModel(
        model_path=pretrained_model_path,
        checkpoint_path_relative="model_checkpoints/latest_validation_checkpoint.pt",
        adaption_strategy="replace_final_block",
        num_classes=num_classes[other_task],
        activation="none"
    )

    checkpoint = torch.load(CKPT_PATH, map_location=device)
    state_dict = checkpoint["model_state_dict"]
    if state_dict and all(k.startswith("module.") for k in state_dict.keys()):
        state_dict = {k[len("module."):]: v for k, v in state_dict.items()}

    model_finetune.load_state_dict(state_dict, strict=True)
    model_finetune = model_finetune.to(device)
    model_finetune.eval()
    # print("✓ Model loaded successfully")
    return model_finetune


In [7]:
import os
import torch
import pandas as pd

from tqdm import tqdm
from torchmetrics import MetricCollection
from metrics import multiclass_segmentation_metrics


all_tasks_list = {
    "BFPDensity_local_T31TFM",
    "BFPDensity_local_T32ULU",
}

BATCH_SIZE = 16
NUM_WORKERS = 8

results_rows = []


for Target_region in all_tasks_list:
    source_region = Target_region

    val_dataset = BuildingCoverageRaster(
        top_dir=TOP_DIR[Target_region],
        s2_tiles=s2_tiles[Target_region],
        labels=labels[Target_region],
        train_val_key="val",
    )

    dataloader = torch.utils.data.DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        drop_last=False,
    )

    model_dir = (
        f"/home/user/results_local/finetuning_results_BFPDensity/{source_region}/LIANet_lr3e-05_batchsize8_nonburned"
    )

    if not os.path.exists(model_dir):
        print(f"Missing folder: {model_dir}")
        continue

    for run_name in sorted(os.listdir(model_dir)):
        ckpt_path = os.path.join(model_dir, run_name, "last.pt")

        if not os.path.exists(ckpt_path):
            continue

        model = load_model(ckpt_path, source_region)
        model.eval()

        list_of_metrics, _ = regression_metrics()

        metrictracker = MetricCollection(list_of_metrics).to(device)

        with torch.no_grad():
            for batch in tqdm(
                dataloader,
                desc=f"{Target_region} | {run_name}",
                leave=False,
            ):
                x = batch["x_s2"].to(device)
                y = batch["y_s2"].to(device)
                label = batch["label"].to(device)
                delta_days = batch["delta_days"].to(device)

                _, pred = model(
                    delta_days,
                    x,
                    y,
                    torch.tensor([0], device=device),
                )

                pred = getattr(pred, "output", pred)

                if pred.dim() == 3:
                    pred = pred.unsqueeze(1)

                metrictracker.update(pred.squeeze(), label)

        results = metrictracker.compute()

        row = {
            "target_region": Target_region,
            "source_region": source_region,
            "seed_or_run": run_name,
            "checkpoint_path": ckpt_path,
        }

        for metric_name, metric_value in results.items():
            row[metric_name] = float(metric_value.detach().cpu())

        results_rows.append(row)

        del model
        del metrictracker
        torch.cuda.empty_cache()

    del dataloader
    del val_dataset
    torch.cuda.empty_cache()


df_local_results = pd.DataFrame(results_rows)

df_local_results.to_csv(
    "BFPDensity_local_baseline_results_all_regions.csv",
    index=False,
)

df_local_results

516 val samples loaded from /home/user/data_shared/T32ULU_val_samples_10perc.json


309 val samples loaded from /home/user/data_shared/T31TFM_val_samples_10perc.json


,target_region,source_region,seed_or_run,checkpoint_path,mae,mse
0,BFPDensity_local_T32ULU,BFPDensity_local_T32ULU,2026-05-11_14-14-33,/home/user/results_local/finetuning_results_BF...,0.117330,0.046552
1,BFPDensity_local_T32ULU,BFPDensity_local_T32ULU,2026-05-11_14-19-44,/home/user/results_local/finetuning_results_BF...,0.118811,0.047475
2,BFPDensity_local_T32ULU,BFPDensity_local_T32ULU,2026-05-11_14-24-43,/home/user/results_local/finetuning_results_BF...,0.116862,0.046215
3,BFPDensity_local_T32ULU,BFPDensity_local_T32ULU,2026-05-11_14-29-41,/home/user/results_local/finetuning_results_BF...,0.118487,0.046696
4,BFPDensity_local_T32ULU,BFPDensity_local_T32ULU,2026-05-11_14-34-38,/home/user/results_local/finetuning_results_BF...,0.115782,0.046039
5,BFPDensity_local_T31TFM,BFPDensity_local_T31TFM,2026-05-11_14-15-03,/home/user/results_local/finetuning_results_BF...,0.124046,0.049720
6,BFPDensity_local_T31TFM,BFPDensity_local_T31TFM,2026-05-11_14-18-46,/home/user/results_local/finetuning_results_BF...,0.124640,0.050389
7,BFPDensity_local_T31TFM,BFPDensity_local_T31TFM,2026-05-11_14-22-27,/home/user/results_local/finetuning_results_BF...,0.125536,0.050039
8,BFPDensity_local_T31TFM,BFPDensity_local_T31TFM,2026-05-11_14-26-15,/home/user/results_local/finetuning_results_BF...,0.126706,0.050562
9,BFPDensity_local_T31TFM,BFPDensity_local_T31TFM,2026-05-11_14-30-05,/home/user/results_local/finetuning_results_BF...,0.125289,0.050166


In [8]:
metadata_cols = [
    "target_region",
    "source_region",
    "seed_or_run",
    "checkpoint_path",
]

metric_cols = [
    col for col in df_local_results.columns
    if col not in metadata_cols
]

# Average over seeds/runs
df_local_seed_avg = (
    df_local_results
    .groupby(["target_region", "source_region"], as_index=False)[metric_cols]
    .mean()
)

# Final average over all local baselines
df_local_final_avg = (
    df_local_seed_avg[metric_cols]
    .mean()
    .to_frame()
    .T
)

df_local_seed_avg.to_csv(
    "BFPDensity_local_baseline_avg_per_region.csv",
    index=False,
)

df_local_final_avg.to_csv(
    "BFPDensity_local_baseline_final_avg_all_regions.csv",
    index=False,
)

df_local_seed_avg, df_local_final_avg

(             target_region            source_region       mae       mse
 0  BFPDensity_local_T31TFM  BFPDensity_local_T31TFM  0.125244  0.050175
 1  BFPDensity_local_T32ULU  BFPDensity_local_T32ULU  0.117454  0.046595,
         mae       mse
 0  0.121349  0.048385)